# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Source: *FlyRank, "The State of AI-Driven SEO in Numbers", March 2026* (`docs/flyrank-seo-research-march-2026.pdf`). The paper is clear about its own standard: direct aggregate comparisons lead, ML is an exploratory appendix, and results are observational. My questions are the ones I'd want asked of my own work, not a grade.

### Finding #1: "The Anatomy of Growing Content" (CONFIRMED)
**Claim:** pages trending *up* are ~38% longer (3.2K vs 2.3K words) and ~20% younger (184 vs 230 days) than pages trending *down*, over 74,187 rising vs 45,272 falling pages. Action: expand thin pages that already earn impressions.

**Where does the label come from?** "Up/down" is `trend_direction`, computed from the **last 30 days vs the previous 30 days of impressions**: a short window, measured at the same snapshot as word count and age.

**My methodology questions:**
1. **Is the comparison confounded by client (site)?** Long-form sites and growing sites may simply be the *same few clients*: a site that is young, actively publishing and writing long articles would make "longer" and "younger" look like growth drivers even if, *within* each site, length makes no difference. Would the 3.2K vs 2.3K gap survive **within-client** comparisons (e.g. a per-client difference, then averaged)?
2. **Does the validation design carry the action?** The finding is cross-sectional (one snapshot), but the action ("expand thin pages → they keep growing") is causal and forward-looking. A forward test would help: take pages' word count *before* a window, then compare trend in the *next* window. Better still, compare pages that were expanded vs similar pages that weren't. Also, "new" pages (<30 days) are excluded from up/down, which leaves age partly selected on the outcome.

### Finding #4: "The Freshness Multiplier" (CONFIRMED)
**Claim:** 365+ day content refreshed within 30 days shows a **3.2× health boost (10.7 → 34.5)** and **57× more impressions (71 → 4,039)**. Action: run a recurring refresh program for mature pages.

**Where does the label come from?** "Refreshed" = `days_since_last_update ≤ 30` at the snapshot. Health score is a FlyRank composite that itself contains impressions (30 points) and position (30 points).

**My methodology questions:**
1. **Which way does the arrow point (selection)?** Editors don't refresh pages at random. They often refresh pages that are *already* doing well or are strategically important. A 57× impressions gap between refreshed and unrefreshed old pages could mostly mean "we refresh the pages that already get traffic". Is there a **before/after on the same pages**, i.e. impressions in the 30 days *before* the refresh vs after, against a matched group of similar old pages that were *not* refreshed?
2. **Is the outcome partly the input?** Health score includes impressions and position, so "3.2× health" and "57× impressions" are not two independent confirmations. Also, the paper itself flags small, survivor-biased cells nearby (the 361+ bucket with 1 declining page). What is the *n* behind the 10.7 → 34.5 comparison, and does it hold across clients rather than in one large one?

Both findings are honestly labelled as observational in the paper. My questions are about how far the "What to do" actions can lean on them.


In [1]:
# The two findings as the paper reports them, plus what a stronger design would need
import pandas as pd
audit = pd.DataFrame([
    {"finding": "#1 Growing content is longer + younger", "reported": "3.2K vs 2.3K words; 184 vs 230 days; n=74,187 up / 45,272 down",
     "label_source": "30d vs prev-30d impressions (same snapshot)", "key_risk": "client confounding; cross-sectional",
     "stronger_design": "within-client comparison; word count measured before the trend window"},
    {"finding": "#4 Refreshed old content: 3.2x health, 57x impressions", "reported": "health 10.7 -> 34.5; impressions 71 -> 4,039",
     "label_source": "days_since_last_update <= 30 at snapshot", "key_risk": "selection (refresh what already works); health contains impressions",
     "stronger_design": "same-page before/after vs matched unrefreshed pages; report n per cell"},
])
pd.set_option("display.max_colwidth", 80)
audit.T


,0,1
finding,#1 Growing content is longer + younger,"#4 Refreshed old content: 3.2x health, 57x impressions"
reported,"3.2K vs 2.3K words; 184 vs 230 days; n=74,187 up / 45,272 down","health 10.7 -> 34.5; impressions 71 -> 4,039"
label_source,30d vs prev-30d impressions (same snapshot),days_since_last_update <= 30 at snapshot
key_risk,client confounding; cross-sectional,selection (refresh what already works); health contains impressions
stronger_design,within-client comparison; word count measured before the trend window,same-page before/after vs matched unrefreshed pages; report n per cell


## 2. My model under an honest split (before/after)

My Week-5 result is "the Week-4 rule ties a Random Forest at P@50 ≈ 0.78 in the ≥500-impression pool". Here I re-run it three ways on the same pages, label and metric:

| Design | What it tests | Honest for my use? |
|---|---|---|
| **BEFORE: random 5-fold by page** | pages of the same site appear in train and test | ✗ the model can memorise each site's "CTR personality" |
| **AFTER 1: GroupKFold(5) by client** (Week 5) | unseen sites, same fortnight | ✓ for "works on a new site" |
| **AFTER 2: time-forward** | rule + model frozen on **March** (decision 16 Mar), scored on **April** (decision 16 Apr) | ✓ for "still works next month", which is how the queue is really used |

The April test is touched once: built by the same function, scored once, and the metrics are written to `work/outputs/w06_validation_metrics.json` as the receipt.


In [2]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, json, subprocess, duckdb, numpy as np, pandas as pd, sklearn
from huggingface_hub import hf_hub_download, get_token
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("work") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

SEED = 42
assert get_token(), "No Hugging Face token: log in (hf auth login) or add HF_TOKEN in Colab Secrets"
REPO = "FlyRank/internship-warehouse"
con = duckdb.connect()
con.execute(f"CREATE VIEW dim_content AS SELECT * FROM read_parquet('{hf_hub_download(REPO, 'dim_content.parquet', repo_type='dataset')}')")
for m in ("2026-03", "2026-04"):
    path = hf_hub_download(REPO, f"fact_content_daily_performance/month={m}/data_0.parquet", repo_type="dataset")
    con.execute(f"CREATE VIEW fact_{m.replace('-', '_')} AS SELECT * FROM read_parquet('{path}')")

def build_frame(month, decision):
    """Same contract as ML-04: features from day 1-15, label from day 16-end of the month."""
    t = f"fact_{month.replace('-', '_')}"
    df = con.sql(f"""
        WITH f AS (SELECT client_hash_id, content_hash_id,
                          SUM(gsc_impressions) AS impr_f, SUM(gsc_clicks) AS clicks_f,
                          SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS pos_f
                   FROM {t} WHERE gsc_data_available IS TRUE AND report_date <  DATE '{decision}' GROUP BY 1, 2),
             o AS (SELECT content_hash_id, SUM(gsc_impressions) AS impr_o, SUM(gsc_clicks) AS clicks_o
                   FROM {t} WHERE gsc_data_available IS TRUE AND report_date >= DATE '{decision}' GROUP BY 1)
        SELECT f.client_hash_id, f.content_hash_id, f.impr_f, f.clicks_f,
               100.0 * f.clicks_f / f.impr_f AS ctr_f, f.pos_f, c.content_type, c.main_intent,
               DATE_DIFF('day', c.content_created_date, DATE '{decision}') AS age_days,
               c.content_updated_date, o.impr_o, 100.0 * o.clicks_o / o.impr_o AS ctr_o
        FROM f JOIN o USING (content_hash_id) LEFT JOIN dim_content c USING (content_hash_id)
        WHERE f.impr_f >= 100 AND o.impr_o >= 100 AND f.pos_f BETWEEN 1 AND 20
    """).df().sort_values("content_hash_id").reset_index(drop=True)
    df["pos_bucket"] = pd.cut(df["pos_f"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
    df["under_captures_next"] = (df["ctr_o"] < 0.5 * df.groupby("pos_bucket", observed=True)["ctr_o"].transform("median")).astype(int)
    df["expected_ctr"] = df.groupby("pos_bucket", observed=True)["ctr_f"].transform("median")
    df["ctr_ratio_f"] = df["ctr_f"] / df["expected_ctr"]
    df["log_impr_f"] = np.log1p(df["impr_f"])
    elig = (df["impr_f"] >= 500) & (df["ctr_ratio_f"] < 0.5)
    df["baseline_score"] = np.where(elig, (df["expected_ctr"] - df["ctr_f"]) / 100 * df["impr_f"], 0.0)
    return df

mar = build_frame("2026-03", "2026-03-16")
apr = build_frame("2026-04", "2026-04-16")
for name, d in [("March (dev)", mar), ("April (time-forward test)", apr)]:
    print(f"{name:<26} {len(d):>6,} pages, {d['client_hash_id'].nunique()} clients, base rate {d['under_captures_next'].mean():.3f}")

NUM = ["log_impr_f", "ctr_f", "pos_f", "age_days", "expected_ctr", "ctr_ratio_f"]
TYPES = sorted(mar["content_type"].dropna().unique())
def X_of(d):
    X = d[NUM].copy()
    X["age_days"] = X["age_days"].fillna(mar["age_days"].median())
    for t in TYPES:
        X[f"type_{t}"] = (d["content_type"] == t).astype(float)
    return X
def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED)
def p_at_k(s, y, k=50):
    o = np.argsort(-np.asarray(s, float), kind="stable"); return np.asarray(y)[o[:k]].mean()

def cv_scores(d, splitter, groups=None):
    X, y = X_of(d), d["under_captures_next"].values
    rows = []
    for tr, te in splitter.split(X, y, groups):
        pool = d["impr_f"].values[te] >= 500
        s_rf = rf().fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        s_bl = d["baseline_score"].values[te]
        for name, s in [("Week-4 rule", s_bl), ("Random Forest", s_rf)]:
            rows.append({"method": name, "P@50": p_at_k(s[pool], y[te][pool]), "AUC": roc_auc_score(y[te], s)})
    return pd.DataFrame(rows).groupby("method").agg(P50=("P@50", "mean"), P50_std=("P@50", "std"), AUC=("AUC", "mean"))

before = cv_scores(mar, KFold(5, shuffle=True, random_state=SEED))
after_g = cv_scores(mar, GroupKFold(5), mar["client_hash_id"].values)

# Time-forward: train once on all of March, score April once
Xm, ym = X_of(mar), mar["under_captures_next"].values
Xa, ya = X_of(apr), apr["under_captures_next"].values
model_mar = rf().fit(Xm, ym)
apr["rf_score"] = model_mar.predict_proba(Xa)[:, 1]
pool_a = apr["impr_f"].values >= 500
fwd = pd.DataFrame([{"method": n, "P50": p_at_k(s[pool_a], ya[pool_a]), "P50_std": np.nan, "AUC": roc_auc_score(ya, s)}
                    for n, s in [("Week-4 rule", apr["baseline_score"].values), ("Random Forest", apr["rf_score"].values)]]).set_index("method")

ba = pd.concat({"BEFORE random-by-page (Mar)": before, "AFTER grouped-by-client (Mar)": after_g,
                "AFTER time-forward (Mar->Apr)": fwd}, names=["design"]).round(3)
ba["pool base rate"] = [mar.loc[mar["impr_f"] >= 500, "under_captures_next"].mean()] * 4 + [ya[pool_a].mean()] * 2
print("\nBefore / after - Precision@50 in the >=500-impression pool, ROC-AUC on all pages:")
print(ba.round(3).to_string())

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w06_validation_metrics.json", "w") as fh:
    json.dump({"seed": SEED, "sklearn": sklearn.__version__,
               "dev": "month=2026-03, decision 2026-03-16", "time_forward_test": "month=2026-04, decision 2026-04-16 (scored once)",
               "results": ba.reset_index().round(4).to_dict(orient="records")}, fh, indent=2, default=str)


March (dev)                57,392 pages, 36 clients, base rate 0.398
April (time-forward test)  59,532 pages, 43 clients, base rate 0.410



Before / after - Precision@50 in the >=500-impression pool, ROC-AUC on all pages:
                                               P50  P50_std    AUC  pool base rate
design                        method                                              
BEFORE random-by-page (Mar)   Random Forest  0.808    0.073  0.811           0.266
                              Week-4 rule    0.732    0.059  0.550           0.266
AFTER grouped-by-client (Mar) Random Forest  0.780    0.062  0.807           0.266
                              Week-4 rule    0.780    0.014  0.547           0.266
AFTER time-forward (Mar->Apr) Week-4 rule    0.880      NaN  0.568           0.295
                              Random Forest  0.900      NaN  0.809           0.295


### What the before/after shows

- **The random split flattered the model.** With pages split at random, the Random Forest scored **P@50 0.808** and appeared to *beat* the rule (0.732). Grouped by client, it drops to **0.780** and **ties** the rule (0.780). That ~3-point gap is the model memorising each site's CTR "personality" from its other pages. Under the random design I'd have wrongly claimed the model wins. (The rule has no fitted weights, so its own movement 0.73 → 0.78 is just different test pages per fold, not memorisation.)
- **Time-forward, both held up.** Frozen on March and scored once on April: rule **0.88** (44/50), forest **0.90** (45/50), base rate 0.295 in the pool. ROC-AUC is stable (0.81 March → 0.81 April). One extra correct page in 50 is not a difference; at n = 50 a single run carries roughly ±0.05 of noise. **The honest reading is still a tie at the top, with the forest better across the full list.**
- April is higher than March for *both* methods, which suggests April's top under-capturers were more persistent, not that anything improved. I would not quote 0.88–0.90 as the expected precision; the 5-fold grouped 0.78 is the more conservative number.

## 3. Leakage audit

The attack checklist from `skills/hunting-leakage-and-validating`, run on my final feature set:

1. **Timeline:** every feature must come from days 1–15; the label from days 16–end.
2. **Single-feature "too good" test:** any one feature with ROC-AUC near 1.0 would be a confession.
3. **Positive control:** add the outcome CTR on purpose and the harness should jump to ~1.0 (it did in ML-04; repeated here on April).
4. **Metadata dates:** `content_created_date` must be before the decision day (age ≥ 0). `content_updated_date` is **not** used: it's an export-time value that can be *after* the decision.
5. **Transductive expectation:** `expected_ctr` is a median over *all* pages, test clients included. Does fitting it on train clients only change anything?
6. **Population selection:** I keep pages with ≥100 impressions in the **outcome** window. That's outcome-window information, so it gets disclosed and measured.


In [3]:
from sklearn.model_selection import GroupKFold
y = mar["under_captures_next"].values
# 2) single-feature AUC (direction-free)
single = {f: max(roc_auc_score(y, X_of(mar)[f]), 1 - roc_auc_score(y, X_of(mar)[f])) for f in NUM}
print("Single-feature ROC-AUC (March):", {k: round(v, 3) for k, v in sorted(single.items(), key=lambda x: -x[1])})

# 3) positive control on April: leak the outcome CTR
Xa_leak = Xa.assign(ctr_o=apr["ctr_o"].values)
leak_auc = roc_auc_score(ya, RandomForestClassifier(n_estimators=100, min_samples_leaf=50, random_state=SEED, n_jobs=-1)
                         .fit(X_of(mar).assign(ctr_o=mar["ctr_o"].values), y).predict_proba(Xa_leak)[:, 1])
print(f"Positive control (outcome CTR leaked): April AUC {leak_auc:.3f}  vs honest {roc_auc_score(ya, apr['rf_score']):.3f}")

# 4) metadata dates
print(f"Pages with creation date AFTER decision (age < 0): March {(mar['age_days'] < 0).sum()}, April {(apr['age_days'] < 0).sum()}")
print(f"content_updated_date after the March decision day: {(pd.to_datetime(mar['content_updated_date']) >= '2026-03-16').mean():.1%} of pages -> excluded from features")

# 5) expected_ctr fitted on train clients only (fold 0 of GroupKFold) vs all pages
tr, te = next(GroupKFold(5).split(mar, groups=mar["client_hash_id"]))
med_train = mar.iloc[tr].groupby("pos_bucket", observed=True)["ctr_f"].median()
med_all = mar.groupby("pos_bucket", observed=True)["ctr_f"].median()
print("Bucket median CTR, train-clients-only vs all pages:", pd.concat({"train_only": med_train, "all": med_all}, axis=1).round(3).to_dict())

# 6) population selection: how many feature-eligible pages drop because of the outcome-window filter?
elig = con.sql("""
    SELECT COUNT(*) FROM (
        SELECT content_hash_id, SUM(gsc_impressions) i, SUM(gsc_avg_position*gsc_impressions)/NULLIF(SUM(gsc_impressions),0) p
        FROM fact_2026_03 WHERE gsc_data_available IS TRUE AND report_date < DATE '2026-03-16' GROUP BY 1)
    WHERE i >= 100 AND p BETWEEN 1 AND 20""").fetchall()[0][0]
print(f"Feature-eligible pages in March: {elig:,}; kept after the outcome-window volume filter: {len(mar):,} "
      f"({len(mar)/elig:.0%}) -> {elig - len(mar):,} pages dropped using outcome information")


Single-feature ROC-AUC (March): {'ctr_f': 0.739, 'ctr_ratio_f': 0.732, 'log_impr_f': 0.705, 'pos_f': 0.578, 'expected_ctr': 0.55, 'age_days': 0.509}


Positive control (outcome CTR leaked): April AUC 0.997  vs honest 0.809
Pages with creation date AFTER decision (age < 0): March 0, April 0
content_updated_date after the March decision day: 82.2% of pages -> excluded from features
Bucket median CTR, train-clients-only vs all pages: {'train_only': {'1-3': 0.248, '4-10': 0.219, '11-20': 0.135}, 'all': {'1-3': 0.268, '4-10': 0.202, '11-20': 0.098}}


Feature-eligible pages in March: 61,467; kept after the outcome-window volume filter: 57,392 (93%) -> 4,075 pages dropped using outcome information


### Leakage audit: verdicts

| Check | Result | Verdict |
|---|---|---|
| Timeline | all features from days 1–15, label from 16–end, by construction in `build_frame` | ✅ clean |
| Single feature "too good" | best single feature `ctr_f` AUC **0.739**; nothing near 1.0 | ✅ clean |
| Positive control | leaking outcome CTR → AUC **0.997** (honest 0.809), so the harness catches leaks | ✅ harness works |
| Product flags | none in the release; none used | ✅ clean |
| Creation date | 0 pages created after the decision day | ✅ clean |
| `content_updated_date` | **82%** of pages show an update *after* 16 March, an export-time value | ⚠️ correctly **excluded**; using it would leak the future |
| `expected_ctr` uses all pages | bucket medians move when fitted on train clients only (e.g. pos 11–20: 0.135 vs 0.098) | ⚠️ **mild transductive leak**; feature-window only (no label info), but should be fitted per fold in the capstone |
| Population filter | **4,075 of 61,467** feature-eligible pages (7%) dropped by needing ≥100 *outcome-window* impressions | ⚠️ **disclosed choice**: the queue is evaluated only on pages that stayed visible, so pages that vanished entirely aren't scored |

## 4. Claim rewrite

My boldest sentences from Weeks 4–5, and what the evidence actually supports:

| Before (too strong) | After (safe language) |
|---|---|
| "The Week-4 rule gets 9 of its top 10 right." (ML-07) | "On one March decision day, **9 of the rule's top 10 were observed** to under-capture in the following two weeks. With n = 10 this is an anecdote, not a precision estimate." |
| "Random Forest matches the rule at P@50 = 0.78 and ranks the whole list far better (AUC 0.81)." (ML-08) | "**Measured** on 5 client-grouped folds of one month, the forest and the rule reached **similar** precision at the top 50 (≈0.78, against a 0.27 pool base rate); the forest **ordered the full list better** (AUC 0.81 vs 0.55). A random page split overstated the forest by ~3 points." |
| "Pages that under-click for their position keep under-clicking." (ML-07) | "**Directionally**, pages far below their position's typical CTR were **more likely** to stay below it two weeks later (67% vs 40% base rate in March); about **a third did not**." |
| "The queue finds pages where a title rewrite will win clicks." (implied, ML-02/03) | "The queue is **decision support**: it lists visible pages whose CTR has been unusually low for their position, so an editor can **review** them first. This data can't show that a rewrite *causes* more clicks; that would need a before/after test with a comparison group." |
| "The method works on new months." | "Frozen on March, it held up **on one later month (April)**: 44/50 for the rule, 45/50 for the forest. One month isn't a trend; May and June would be the next checks, with June kept sealed." |

**The failure examples below** (April's top-50 misses) are mostly the familiar shape: huge impressions (28k–103k), very low CTR, and informational intent. Next fortnight they recovered toward normal CTR, e.g. one page went from 0.049% to 0.932%. That fits a query-mix shift rather than a fixed snippet problem, which is a reason to keep the label as "review", never "broken".

In [4]:
# Failure examples from the time-forward test: what the rule flagged in April that did NOT under-capture
apr["rule_rank"] = apr["baseline_score"].rank(ascending=False, method="first")
top50 = apr[pool_a].nlargest(50, "baseline_score")
misses = top50[top50["under_captures_next"] == 0]
print(f"April top-50 by the frozen rule: {int(top50['under_captures_next'].sum())}/50 right, {len(misses)} misses")
print(misses[["content_type", "main_intent", "pos_f", "impr_f", "ctr_f", "expected_ctr", "ctr_o"]]
      .rename(columns={"ctr_o": "ctr_next (check only)"}).round(3).head(6).to_string(index=False))
print(f"\nClients behind April's top 50: {top50['client_hash_id'].nunique()} | largest share: "
      f"{top50['client_hash_id'].value_counts(normalize=True).iloc[0]:.0%}")


April top-50 by the frozen rule: 44/50 right, 6 misses
   content_type   main_intent  pos_f   impr_f  ctr_f  expected_ctr  ctr_next (check only)
keyword article transactional  5.107 103489.0  0.080         0.165                  0.116
keyword article informational  2.186  43058.0  0.049         0.179                  0.932
keyword article informational  6.035  28631.0  0.021         0.165                  0.163
keyword article informational  3.789  30245.0  0.030         0.165                  0.224
keyword article informational  3.754  37134.0  0.057         0.165                  0.135
keyword article transactional  6.752  38455.0  0.075         0.165                  0.098

Clients behind April's top 50: 12 | largest share: 32%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.